In [2]:
import pandas as pd

df = pd.read_csv("data/cars_synthetic.csv")

df.head()

,car_id,brand,model,year,mileage,horsepower,price
0,1,Renault,Clio,2016,133403,151,50500
1,2,Toyota,Yaris,2021,55882,129,73500
2,3,Toyota,Yaris,2025,8304,118,85000
3,4,Peugeot,308,2016,91451,119,45600
4,5,Citroen,C4,2012,185858,76,11600


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   car_id      500 non-null    int64 
 1   brand       500 non-null    object
 2   model       500 non-null    object
 3   year        500 non-null    int64 
 4   mileage     500 non-null    int64 
 5   horsepower  500 non-null    int64 
 6   price       500 non-null    int64 
dtypes: int64(5), object(2)
memory usage: 27.5+ KB


In [4]:
df.describe()

,car_id,year,mileage,horsepower,price
count,500.000000,500.000000,500.000000,500.000000,500.000000
mean,250.500000,2016.554000,147297.610000,124.878000,43827.800000
std,144.481833,5.081213,90582.229432,33.484407,26431.611137
min,1.000000,2008.000000,2000.000000,65.000000,8000.000000
25%,125.750000,2012.000000,76347.500000,96.000000,19600.000000
50%,250.500000,2017.000000,138249.500000,125.000000,43300.000000
75%,375.250000,2021.000000,210230.250000,155.000000,64475.000000
max,500.000000,2025.000000,391532.000000,180.000000,103500.000000


In [5]:
X = df[["year", "mileage", "horsepower"]]
y = df["price"]

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(400, 3)
(100, 3)
(400,)
(100,)


In [9]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [10]:
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
y_pred = model.predict(X_test)

In [12]:
from sklearn.metrics import mean_absolute_error

In [13]:
mae = mean_absolute_error(y_test, y_pred)

In [14]:
mae

3986.66

In [15]:
y_pred_train = model.predict(X_train)
mae_train = mean_absolute_error(y_train, y_pred_train)

In [16]:
print("MAE Train :", mae_train)
print("MAE Test  :", mae)

MAE Train : 1348.3075
MAE Test  : 3986.66


In [17]:
results = X_test.copy()

results["price_real"] = y_test
results["price_pred"] = y_pred

results["error"] = abs(
    results["price_real"] - results["price_pred"]
)

results.head(10)

,year,mileage,horsepower,price_real,price_pred,error
361,2021,58410,147,72300,74703.0,2403.0
73,2010,201474,107,10200,11650.0,1450.0
374,2016,195129,118,29100,30212.0,1112.0
155,2011,305602,121,8000,8097.0,97.0
104,2022,64334,68,64400,59244.0,5156.0
394,2019,76283,74,53500,54209.0,709.0
377,2013,275102,157,15800,15552.0,248.0
124,2015,145924,123,28300,37768.0,9468.0
68,2011,323430,150,8000,8385.0,385.0
450,2009,234425,175,10100,12390.0,2290.0


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [19]:
numeric_features = ["year", "mileage", "horsepower"]
categorical_features = ["brand", "model"]

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [21]:
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=8,
                random_state=42
            )
        )
    ]
)

In [22]:
X = df[
    [
        "brand",
        "model",
        "year",
        "mileage",
        "horsepower"
    ]
]

y = df["price"]

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [24]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
y_pred = pipeline.predict(X_test)

In [26]:
mae = mean_absolute_error(y_test, y_pred)

print("MAE Test :", mae)

MAE Test : 3954.2275329935846


In [27]:
y_pred_train = pipeline.predict(X_train)
mae_train = mean_absolute_error(y_train, y_pred_train)

In [28]:
print("MAE Train :", mae_train)
print("MAE Test  :", mae)

MAE Train : 1612.8139472048183
MAE Test  : 3954.2275329935846


In [29]:
from sklearn.model_selection import cross_val_score

In [30]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)

In [31]:
scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)

mae_cv = -scores

print(mae_cv)
print("MAE CV moyenne :", mae_cv.mean())

[3290.075 4713.875 3781.3   3766.4   3239.025]
MAE CV moyenne : 3758.1349999999998


In [32]:
from sklearn.model_selection import GridSearchCV

In [33]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_leaf": [1, 2, 5]
}

In [34]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ]
)

In [35]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

In [36]:
grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [None, 8, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__n_estimators': [100, 200]}"
,scoring,'neg_mean_absolute_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...)]"


In [37]:
print("Meilleurs paramètres :", grid_search.best_params_)
print("Meilleure MAE CV :", -grid_search.best_score_)

Meilleurs paramètres : {'model__max_depth': 12, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}
Meilleure MAE CV : 3741.376564475693


In [38]:
best_model = grid_search.best_estimator_

In [39]:
y_pred_test = best_model.predict(X_test)

mae_test = mean_absolute_error(
    y_test,
    y_pred_test
)

print("MAE CV   :", -grid_search.best_score_)
print("MAE Test :", mae_test)

MAE CV   : 3741.376564475693
MAE Test : 3944.422985403485


In [40]:
import joblib

In [42]:
joblib.dump(
    best_model,
    "models/car_price_model.pkl"
)

['models/car_price_model.pkl']

In [43]:
loaded_model = joblib.load(
    "models/car_price_model.pkl"
)

In [44]:
loaded_model.predict(X_test.head(5))

array([74066.44761905, 12927.96245421, 29163.38394661,  8335.        ,
       58137.375     ])